In [14]:
!apt-get install -y coinor-cbc

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  coinor-libcbc3 coinor-libcgl1 coinor-libclp1 coinor-libcoinutils3v5
  coinor-libosi1v5
The following NEW packages will be installed:
  coinor-cbc coinor-libcbc3 coinor-libcgl1 coinor-libclp1
  coinor-libcoinutils3v5 coinor-libosi1v5
0 upgraded, 6 newly installed, 0 to remove and 2 not upgraded.
Need to get 2,908 kB of archives.
After this operation, 8,310 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 coinor-libcoinutils3v5 amd64 2.11.4+repack1-2 [465 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 coinor-libosi1v5 amd64 0.108.6+repack1-2 [275 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 coinor-libclp1 amd64 1.17.5+repack1-1 [937 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 coinor-libcgl1 amd64 0.60.3+repack1-3 [420 kB]
Get:5 htt

In [3]:
import pulp
import shutil
print(shutil.which("cbc"))

None


In [4]:
departments = [
    "Ophthalmology",
    "Gynecology",
    "Oral Surgery",
    "Otolaryngology",
    "General Surgery"
]

days = ["Mon", "Tue", "Wed", "Thu", "Fri"]

In [5]:
ORs_per_day = 10
hours_per_OR = 8

In [6]:
# Total surgery teams available per day (n_k)
nk = {
    "Ophthalmology": {"Mon":2, "Tue":2, "Wed":2, "Thu":2, "Fri":2},
    "Gynecology": {"Mon":3, "Tue":3, "Wed":3, "Thu":3, "Fri":3},
    "Oral Surgery": {"Mon":0, "Tue":1, "Wed":0, "Thu":1, "Fri":0},
    "Otolaryngology": {"Mon":1, "Tue":1, "Wed":1, "Thu":1, "Fri":1},
    "General Surgery": {"Mon":6, "Tue":6, "Wed":6, "Thu":6, "Fri":6}
}

In [7]:
# Weekly target hours
target_hours = {
    "Ophthalmology": 39.4,
    "Gynecology": 117.4,
    "Oral Surgery": 19.9,
    "Otolaryngology": 26.3,
    "General Surgery": 189
}

# Max ORs per department per day
max_daily = {
    "Ophthalmology": {"Mon":2, "Tue":2, "Wed":2, "Thu":2, "Fri":2},
    "Gynecology": {"Mon":3, "Tue":3, "Wed":3, "Thu":3, "Fri":3},
    "Oral Surgery": {"Mon":1, "Tue":1, "Wed":1, "Thu":1, "Fri":1},
    "Otolaryngology": {"Mon":1, "Tue":1, "Wed":1, "Thu":1, "Fri":1},
    "General Surgery": {"Mon":6, "Tue":6, "Wed":6, "Thu":6, "Fri":6}
}

# Weekly min / max
min_weekly = {
    "Ophthalmology": 3,
    "Gynecology": 12,
    "Oral Surgery": 2,
    "Otolaryngology": 2,
    "General Surgery": 18
}

max_weekly = {
    "Ophthalmology": 6,
    "Gynecology": 18,
    "Oral Surgery": 3,
    "Otolaryngology": 4,
    "General Surgery": 25
}

In [8]:
model = pulp.LpProblem("OR_Scheduling", pulp.LpMaximize)

In [9]:
# Decision variables
x = pulp.LpVariable.dicts(
    "ORs",
    [(j, k) for j in departments for k in days],
    lowBound=0,
    cat="Integer"
)


In [10]:
#Objective function
model += pulp.lpSum(
    hours_per_OR * x[(j, k)] / target_hours[j]
    for j in departments for k in days
)

In [11]:
#constraints
# (1) Physical OR capacity: At most 10 ORs are assigned every day
for k in days:
    model += pulp.lpSum(x[(j, k)] for j in departments) <= ORs_per_day

# (2) Surgery team capacity: The number of ORs allocated to a department on a
#given day cannot exceed the number of surgery
#teams that department has available that day
for j in departments:
    for k in days:
        model += x[(j, k)] <= nk[j][k]

# (3) Daily min / max per department: Meet department daily minimums and maximums
for j in departments:
    for k in days:
        model += x[(j, k)] <= max_daily[j][k]

# (4) Weekly min / max per department: Meet department weekly minimums and maximums
for j in departments:
    model += pulp.lpSum(x[(j, k)] for k in days) >= min_weekly[j]
    model += pulp.lpSum(x[(j, k)] for k in days) <= max_weekly[j]

# (5) Weekly assigned hours cannot exceed target hours
for j in departments:
    model += pulp.lpSum(x[(j, k)] * hours_per_OR for k in days) <= target_hours[j]

#MODIFICA SOLUZIONE PER BILANCIAMENTO
# (6) General Surgery: minimum 4 OR per day
#for k in days:
    #model += x[("General Surgery", k)] >= 4

# (7) Ophthalmology: maximum 1 OR per day
#for k in days:
    #model += x[("Ophthalmology", k)] <= 1

#Modifica ulteriore ottimizzazione general surgery
#(8) General Surgery: minimum 5 OR per day
#for k in days:
 #   model += x[("General Surgery", k)] >= 5

In [15]:
 # model.solve()

solver = pulp.COIN_CMD(
    timeLimit=30,
    msg=True
)

model.solve(solver)

1

In [ ]:
print("Optimal OR allocation:\n")
for j in departments:
    for k in days:
        val = x[(j, k)].value()
        if val > 0:
            print(f"{j:18s} {k}: {int(val)}")

Optimal OR allocation:

Ophthalmology      Wed: 2
Ophthalmology      Thu: 2
Gynecology         Mon: 3
Gynecology         Tue: 3
Gynecology         Wed: 3
Gynecology         Thu: 3
Gynecology         Fri: 2
Oral Surgery       Tue: 1
Oral Surgery       Thu: 1
Otolaryngology     Mon: 1
Otolaryngology     Tue: 1
Otolaryngology     Fri: 1
General Surgery    Mon: 6
General Surgery    Tue: 4
General Surgery    Wed: 5
General Surgery    Thu: 2
General Surgery    Fri: 6


In [16]:
print("\nSCHEDULE TABLE\n")

# intestazione
header = "Department        " + " ".join(f"{k:>5s}" for k in days)
print(header)
print("-" * len(header))

# righe
for j in departments:
    row = f"{j:18s}"
    for k in days:
        val = int(x[(j, k)].value())
        row += f"{val:5d}"
    print(row)


SCHEDULE TABLE

Department          Mon   Tue   Wed   Thu   Fri
-----------------------------------------------
Ophthalmology         0    0    2    2    0
Gynecology            3    3    3    2    3
Oral Surgery          0    1    0    1    0
Otolaryngology        1    1    0    0    1
General Surgery       5    5    5    4    4


In [17]:
print("\nObjective value:", pulp.value(model.objective))


Objective value: 4.456298750836373


In [18]:
print("\nWEEKLY TOTALS\n")

for j in departments:
    weekly_ors = sum(x[(j, k)].value() for k in days)
    weekly_hours = hours_per_OR * weekly_ors
    perc_target = weekly_hours / target_hours[j] * 100

    print(f"{j:18s} | ORs/week: {int(weekly_ors):2d} | Hours/week: {weekly_hours:5.1f}| % target: {perc_target:6.1f}%")



WEEKLY TOTALS

Ophthalmology      | ORs/week:  4 | Hours/week:  32.0| % target:   81.2%
Gynecology         | ORs/week: 14 | Hours/week: 112.0| % target:   95.4%
Oral Surgery       | ORs/week:  2 | Hours/week:  16.0| % target:   80.4%
Otolaryngology     | ORs/week:  3 | Hours/week:  24.0| % target:   91.3%
General Surgery    | ORs/week: 23 | Hours/week: 184.0| % target:   97.4%
